# 1. Data Pre-processing and exploration

In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score,confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import EarlyStopping 

In [ ]:
# Combining all the train files into one dataframw
train_folder_path = "./MLT-CW-Dataset"

train_file_names = [f for f in os.listdir(train_folder_path) if f.endswith('.csv') and os.path.isfile(os.path.join(train_folder_path, f))]
train_file_paths = [os.path.join(train_folder_path, name) for name in train_file_names]

train_df_list = []

for file_path in train_file_paths:
    current_df = pd.read_csv(file_path)
    current_df['source'] = file_path
    train_df_list.append(current_df)
train = pd.concat(train_df_list, axis=0, ignore_index=True)
train

In [ ]:
# Combining all the test files into one dataframw
test_folder_path = "./MLT-CW-Dataset/test-set"
test_file_names = [f for f in os.listdir(test_folder_path) if f.endswith('.csv') and os.path.isfile(os.path.join(test_folder_path, f))]
test_file_paths = [os.path.join(test_folder_path, name) for name in test_file_names]
test_df_list = []

for file_path in test_file_paths:
    current_df = pd.read_csv(file_path)
    current_df['source'] = file_path

    test_df_list.append(current_df)
test = pd.concat(test_df_list, axis=0, ignore_index=True)
test

In [ ]:
test.drop(['index'],axis=1,inplace=True)
train.drop(['index','Unnamed: 0'],axis=1,inplace=True)

In [ ]:
train

In [ ]:
test

### a. Report the class balance of whole dataset

In [ ]:
colums_to_merge=['timestamp','back_x','back_y','back_z','thigh_x','thigh_y','thigh_z','label','source']
train_and_test=pd.merge(train, test, on=colums_to_merge,how='outer')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
class_balance = train_and_test['label'].value_counts().sort_index()

plt.figure(figsize=(14, 7))
bars = plt.bar(x=class_balance.index.astype(str), height=class_balance.values)
plt.title('Class Distribution by Activity Type', fontsize=16)
plt.xlabel('Class', fontsize=14)
plt.ylabel('No of rows(in 10^6)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 5, int(yval), ha='center', va='bottom', fontsize=10)

plt.ylim(0, class_balance.values.max() * 1.15)
plt.tight_layout()
plt.savefig('class_distribution_by_activity_name.png')

### b. Remove all data related cycling from the dataset

In [ ]:
train_update=train.copy()
test_update=test.copy()

In [ ]:
train_update['label'] = train_update['label'].astype(int)
test_update['label'] = test_update['label'].astype(int)

In [ ]:
def drop_cycling_data(df):
    labels_to_remove = [13, 14, 130, 140]
    # df['label'].isin(labels_to_remove) will mark true where the value of lable to remove is equal 13,14,130 and 140 as true 
    # ~ sign will change true to False and False to True
    mast_to_keep = ~df['label'].isin(labels_to_remove)
    df=df[mast_to_keep].copy()
    return df

In [ ]:
train_update=drop_cycling_data(train_update)

In [ ]:
test_update=drop_cycling_data(test_update)

In [ ]:
print(f"No of rows remaining after Removing all cycling Data from the Test Dataset: {train_update['label'].count()} rows")

In [ ]:
print(f"No of rows remaining after Removing all cycling Data from the Test Dataset: {test_update['label'].count()} rows")


### c. Mearge stairs ascending descending into one class and name it 9

In [ ]:
def combining_strais_data(df):
    df['label'].replace([4,5],9,inplace=True)
    return df

In [ ]:
train_update=combining_strais_data(train_update)

In [ ]:
test_update=combining_strais_data(test_update)

### d. Sampling and visualizing

In [ ]:
train_and_test_after_update=pd.merge(train_update, test_update, on=colums_to_merge,how='outer')
walking_data=train_and_test_after_update[train_and_test_after_update['label'] == 1]

# Sampling the walking data
walking_data_sampling=walking_data[:1000]

# Visualizing sampled walking data
plt.scatter(x=walking_data_sampling['back_x'],y=walking_data_sampling['thigh_x'],c='skyblue')
plt.title('Walking')
plt.xlabel('x-axis coordinates of back')
plt.ylabel('x-axis coordinates of thigh')
plt.show()

plt.scatter(x=walking_data_sampling['back_y'],y=walking_data_sampling['thigh_y'],c='orange')
plt.title('Walking')
plt.xlabel('y-axis coordinates of back')
plt.ylabel('y-axis coordinates of thigh')
plt.show()

plt.scatter(x=walking_data_sampling['back_z'],y=walking_data_sampling['thigh_z'],c='green')
plt.title('Walking')
plt.xlabel('z-axis coordinates of back')
plt.ylabel('z-axis coordinates of thigh')
plt.show()

# 2.	Data Cleaning and Preparation 

### a. Finding probelm in dataset S007

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Removing class 10 from the S007.csv file
label_10_data = train_and_test_after_update[train_and_test_after_update['label'] == 10]
sources_with_label_10 = label_10_data['source'].unique()
counts_per_source = label_10_data['source'].value_counts()

plt.figure(figsize=(6, 6))
bars = plt.bar(counts_per_source.index, counts_per_source.values, color='teal')
plt.title('Count of Activity Label 10 Rows per Source File')
plt.xlabel('Source File')
plt.ylabel('Count of Rows (Label 10)')
plt.xticks(ha='center')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 5, int(yval), ha='center', va='bottom', fontsize=10)

plt.ylim(0, counts_per_source.values.max() * 1.15)
plt.savefig('label_10_count_per_source.png')

### b. imputation strategy for low quality data in S007

In [ ]:
import pandas as pd
indices_to_drop = train_update[train_update['label'] == 10].index
indices_to_drop
train_update=train_update.drop(train_update[train_update['label'] == 10].index)

In [ ]:
print(f"Size of dataset after cleaning S007.csv File: {train_update['label'].count()} rows")

### c. Sliding Window

In [ ]:
train_update.drop(['source'],axis=1,inplace=True)
test_update.drop(['source'],axis=1,inplace=True)

In [ ]:
import numpy as np
import pandas as pd
# --- 1. Main Function to Create Windows and Labels (Optimized) ---

def create_sliding_windows_optimized(df, Fs, time_window_sec, step_time_sec):

    window_size_points = int(time_window_sec * Fs)
    step_size_points = int(step_time_sec * Fs)

    windows = []
    labels = []
    start_index = 0
    total_len = len(df)

    while start_index + window_size_points <= total_len:
        
        end_index = start_index + window_size_points
        window = df.iloc[start_index : end_index].copy()
        
        if len(window) > 1:
            windows.append(window)
            
            window_label = window['label'].mode()[0]
            labels.append(window_label)
        
        start_index += step_size_points
            
    return windows, labels

train_df_to_process = train_update.copy()
test_df_to_process = test_update.copy()

train_windows, train_labels = create_sliding_windows_optimized(train_df_to_process, 50, 2.0, 1.0)
test_windows, test_labels = create_sliding_windows_optimized(test_df_to_process, 50, 2.0, 1.0)
   
print('Training DataSet:')
print(f"Total original data points (rows): {len(train_df_to_process)}")
print(f"Total windows generated: {len(train_windows)}")
print(f"Total labels generated: {len(train_labels)}")

print('\nTesting DataSet:')
print(f"Total original data points (rows): {len(test_df_to_process)}")
print(f"Total windows generated: {len(test_windows)}")
print(f"Total labels generated: {len(test_labels)}")

In [ ]:
def add_baseline_features(df):
    df['back_EN'] = np.sqrt(df['back_x']**2 + df['back_y']**2 + df['back_z']**2)
    
    df['back_ENMO'] = np.maximum(0, df['back_EN'] - 1)
    
    df['thigh_EN'] = np.sqrt(df['thigh_x']**2 + df['thigh_y']**2 + df['thigh_z']**2)
    
    df['thigh_ENMO'] = np.maximum(0, df['thigh_EN'] - 1)
    
    return df

def apply_features_to_windows(windows_list):
    processed_windows = [add_baseline_features(window) for window in windows_list]
    return processed_windows

train_windows_copy=train_windows.copy()
test_windows_copy=test_windows.copy()

train_windows_processed = apply_features_to_windows(train_windows_copy)
test_windows_processed = apply_features_to_windows(test_windows_copy)

# 3. Pipelines 

### a. Baseline

Note: I am having deficulties while installing seaborn in my device. Because of that i have used GenAI to genrate the code for ploting confusion matrix.

In [ ]:
# Standardization
def standardize_windows(train_windows, test_windows, feature_columns):

    all_train_features = pd.concat(train_windows)[feature_columns]
    scaler = StandardScaler()
    scaler.fit(all_train_features)
    def transform_window(window_df):
        #
        features_transformed = scaler.transform(window_df[feature_columns])
        return features_transformed

    X_train_list = [transform_window(w) for w in train_windows]
    X_train_scaled = np.array(X_train_list)

    # Apply to all testing windows and stack them for CNN input
    X_test_list = [transform_window(w) for w in test_windows]
    X_test_scaled = np.array(X_test_list)

    return X_train_scaled, X_test_scaled, scaler

FEATURE_COLUMNS = ['back_x', 'back_y', 'back_z','thigh_x', 'thigh_y', 'thigh_z','back_EN', 'back_ENMO', 'thigh_EN', 'thigh_ENMO']

X_train_scaled, X_test_scaled, scaler = standardize_windows(
    train_windows_processed,
    test_windows_processed,
    FEATURE_COLUMNS
)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

In [ ]:
# Genrate report of Precision,recall and f1-score per class

def generate_final_report(y_test_original_labels, y_pred_original_labels, class_names_map, digits=4):
    print("\n--- Generating Predictions on Test Data ---")
    
    present_labels = sorted(np.unique(y_test_original_labels))
    
    target_names = [class_names_map[label] for label in present_labels]   

    print("--- Calculating Classification Report ---")
    
    report = classification_report(
        y_true=y_test_original_labels,      
        y_pred=y_pred_original_labels,       
        labels=present_labels,
        target_names=target_names, 
        digits=digits
    )

    print("\n### Per-Class Classification Report ###")
    print(report)

ACTIVITY_NAMES = {
    1: 'walking',
    2: 'running',
    3: 'shuffling',
    6: 'standing',
    7: 'sitting',
    8: 'lying',
    9: 'stairs'    
}


#### CNN Baseline

In [ ]:
def build_baseline_cnn_model(sequence_length, num_features, num_classes):
    input_shape = (sequence_length, num_features)
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv1D(filters=32, kernel_size=5, activation='relu', input_shape=input_shape,name='Conv1D_1'),
        tf.keras.layers.MaxPooling1D(pool_size=2,name='MaxPool1'),
        tf.keras.layers.Flatten(name='Flatten_Features'),
        tf.keras.layers.Dense(64, activation='relu',name='Dense_1'),
        tf.keras.layers.Dense(num_classes, activation='softmax',name='Output_Layer')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'] 
    )    
    return model

In [ ]:
SEQUENCE_LENGTH = X_train_scaled.shape[1] 
NUM_FEATURES = X_train_scaled.shape[2]

encoder=LabelEncoder()
y_train_encoded=encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)
NUM_CLASSES = len(encoder.classes_)
y_train_final = y_train_encoded
y_test_final = y_test_encoded

class_labels = list(ACTIVITY_NAMES.values())

In the cell below i have have accidently named the confusion matrix "Confusion Matrix for Improved CNN Model" insetad of "Confusion Matrix for Baseline CNN Model"

In [ ]:
cnn_base_model = build_baseline_cnn_model(SEQUENCE_LENGTH, NUM_FEATURES, NUM_CLASSES)
history = cnn_base_model.fit(X_train_scaled,y_train_final,epochs=10,validation_split=0.2,verbose=1)

# For claculation the Precision, recall per class
y_pred_proba = cnn_base_model.predict(X_test_scaled)
y_pred_classes_encoded = np.argmax(y_pred_proba, axis=1)
y_pred_classes_original = encoder.inverse_transform(y_pred_classes_encoded)
generate_final_report(y_test_original_labels=y_test, y_pred_original_labels=y_pred_classes_original,class_names_map=ACTIVITY_NAMES)


train_loss_cnn_base, train_accuracy_cnn_base = cnn_base_model.evaluate(X_train_scaled, y_train_final, verbose=0)

conf_matrix_cnn_base = confusion_matrix(y_test, y_pred_classes_original)

fig_cnn_base, ax_cnn_base = plt.subplots(figsize=(10, 7))
im = ax_cnn_base.imshow(conf_matrix_cnn_base, interpolation='nearest', cmap=plt.cm.Purples)
fig_cnn_base.colorbar(im)


ax_cnn_base.set(
    xticks=np.arange(len(class_labels)),
    yticks=np.arange(len(class_labels)),
    xticklabels=class_labels,
    yticklabels=class_labels,
    title='Confusion Matrix for Improved CNN Model',
    ylabel='True Activity Label',
    xlabel='Predicted Activity Label'
)

plt.setp(ax_cnn_base.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh_cnn_base = conf_matrix_cnn_base.max() / 2.
for i in range(conf_matrix_cnn_base.shape[0]):
    for j in range(conf_matrix_cnn_base.shape[1]):
        ax_cnn_base.text(
            j, i, conf_matrix_cnn_base[i, j],
            ha="center", va="center",
            color="white" if conf_matrix_cnn_base[i, j] > thresh_cnn_base else "black"
        )

plt.tight_layout()
plt.show()

#### K-Means BaseLine

In [ ]:
# Store the original shapes for reference
n_samples_train, seq_len, n_features = X_train_scaled.shape
n_samples_test, _, _ = X_test_scaled.shape

# Reshape the data from (N, 100, 10) to (N, 1000)
X_train_2d = X_train_scaled.reshape(n_samples_train, seq_len * n_features)
X_test_2d = X_test_scaled.reshape(n_samples_test, seq_len * n_features)

print(f"Original 3D Shape: {X_train_scaled.shape}")
print(f"Reshaped 2D Shape: {X_train_2d.shape}")

N_CLUSTERS = NUM_CLASSES 

print(f"\nTraining K-Means with {N_CLUSTERS} clusters...")
kmeans = KMeans(
    n_clusters=N_CLUSTERS, 
    random_state=42, 
    n_init=10, 
    max_iter=300
) 

kmeans.fit(X_train_2d)

y_kmeans_test_pred = kmeans.predict(X_test_2d)

print("\n--- K-Means Evaluation (Clustering Metrics) ---")

labels_encoded = y_train_final
clusters = kmeans.labels_

mapping = {}
for cluster_id in range(N_CLUSTERS):
    labels_in_cluster = labels_encoded[clusters == cluster_id]
    
    if len(labels_in_cluster) > 0:
        majority_label = pd.Series(labels_in_cluster).value_counts().index[0]
        mapping[cluster_id] = majority_label
    else:
        mapping[cluster_id] = 0

print("\nCluster-to-Label Mapping (Cluster ID -> True Encoded Label):")
print(mapping)

y_mapped_test_pred_encoded = np.array([mapping.get(pred, 0) for pred in y_kmeans_test_pred])


y_pred_classes_original_kmeans = encoder.inverse_transform(y_mapped_test_pred_encoded)

print("\n--- Generating Classification Report for K-Means Baseline ---")
generate_final_report(
    y_test_original_labels=y_test, 
    y_pred_original_labels=y_pred_classes_original_kmeans,
    class_names_map=ACTIVITY_NAMES
)

class_labels = list(ACTIVITY_NAMES.values())
conf_matrix = confusion_matrix(y_test, y_pred_classes_original_kmeans)

fig, ax = plt.subplots(figsize=(10, 7))

im = ax.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Greens)
fig.colorbar(im)

ax.set(
    xticks=np.arange(len(class_labels)),
    yticks=np.arange(len(class_labels)),
    xticklabels=class_labels,
    yticklabels=class_labels,
    title='Confusion Matrix for K-Means Clustering (Mapped)',
    ylabel='True Activity Label',
    xlabel='Predicted Activity Label'
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = conf_matrix.max() / 2.
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        ax.text(
            j, i, conf_matrix[i, j],
            ha="center", va="center",
            color="white" if conf_matrix[i, j] > thresh else "black"
        )

plt.tight_layout()
plt.show()


#### Random Forest Base Line

In [ ]:
print("--- Initializing and Training Random Forest Classifier ---")
rf_model = RandomForestClassifier(
    n_estimators=10,             
    random_state=24          
)

rf_model.fit(X_train_2d, y_train_final)
y_pred_encoded_rf = rf_model.predict(X_test_2d)
y_pred_classes_original_rf = encoder.inverse_transform(y_pred_encoded_rf)

print("\n--- Generating Classification Report for Random Forest Baseline ---")

generate_final_report(y_test_original_labels=y_test, y_pred_original_labels=y_pred_classes_original_rf,class_names_map=ACTIVITY_NAMES)

# Ploting confusion matrix
class_labels = list(ACTIVITY_NAMES.values())
conf_matrix = confusion_matrix(y_test, y_pred_classes_original_rf)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
fig.colorbar(im)
ax.set(
    xticks=np.arange(len(class_labels)),
    yticks=np.arange(len(class_labels)),
    xticklabels=class_labels,
    yticklabels=class_labels,
    title='Confusion Matrix for Random Forest Classifier',
    ylabel='True Activity Label',
    xlabel='Predicted Activity Label'
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = conf_matrix.max() / 2.
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        ax.text(
            j, i, conf_matrix[i, j],
            ha="center", va="center",
            color="white" if conf_matrix[i, j] > thresh else "black"
        )
plt.tight_layout()
plt.show()

### b. FineTuning

#### CNN Fine Tuning

In [ ]:
def build_baseline_cnn_model_improved(sequence_length, num_features, num_classes):
    input_shape = (sequence_length, num_features)
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=input_shape,name='Conv1D_1'),
        tf.keras.layers.Conv1D(filters=128, kernel_size=3, activation='relu',name='Conv1D_2'),
        tf.keras.layers.MaxPooling1D(pool_size=2,name='MaxPool1'),
        tf.keras.layers.Flatten(name='Flatten_Features'),
        tf.keras.layers.Dense(128, activation='relu',name='Dense_1'),
        tf.keras.layers.Dense(64, activation='relu',name='Dense_2'),
        tf.keras.layers.Dense(num_classes, activation='softmax',name='Output_Layer')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'] 
    )    
    return model

In [ ]:
model = build_baseline_cnn_model_improved(SEQUENCE_LENGTH, NUM_FEATURES, NUM_CLASSES)
history_improved = model.fit(X_train_scaled,y_train_final,epochs=10,validation_split=0.2,verbose=1)
score_improved = model.evaluate(X_test_scaled,y_test_final, verbose=0)
y_pred_proba_improved = model.predict(X_test_scaled)
y_pred_classes_encoded__improved = np.argmax(y_pred_proba_improved, axis=1)
y_pred_classes_original__improved = encoder.inverse_transform(y_pred_classes_encoded__improved)

# generating report  
generate_final_report(y_test_original_labels=y_test, y_pred_original_labels=y_pred_classes_original__improved,class_names_map=ACTIVITY_NAMES)

# Ploting confusion matrix
class_labels = list(ACTIVITY_NAMES.values())
conf_matrix = confusion_matrix(y_test, y_pred_classes_original__improved)

fig, ax = plt.subplots(figsize=(10, 7))

im = ax.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Purples) 
fig.colorbar(im)

ax.set(
    xticks=np.arange(len(class_labels)),
    yticks=np.arange(len(class_labels)),
    xticklabels=class_labels,
    yticklabels=class_labels,
    title='Confusion Matrix for Improved CNN Model',
    ylabel='True Activity Label',
    xlabel='Predicted Activity Label'
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = conf_matrix.max() / 2.
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        ax.text(
            j, i, conf_matrix[i, j],
            ha="center", va="center",
            color="white" if conf_matrix[i, j] > thresh else "black"
        )

plt.tight_layout()
plt.show()

### K-means Fine Tuning

In [ ]:
def find_mode_numpy(arr):
    unique_labels, counts = np.unique(arr, return_counts=True)
    max_count_index = np.argmax(counts)
    return unique_labels[max_count_index]

n_samples_train, seq_len, n_features = X_train_scaled.shape
n_samples_test, _, _ = X_test_scaled.shape

X_train_2d = X_train_scaled.reshape(n_samples_train, seq_len * n_features)
X_test_2d = X_test_scaled.reshape(n_samples_test, seq_len * n_features)

N_CLUSTERS = NUM_CLASSES 

kmeans = KMeans(n_clusters=N_CLUSTERS,random_state=24,n_init=15,max_iter=500) 

kmeans.fit(X_train_2d)

y_kmeans_test_pred = kmeans.predict(X_test_2d)

labels_encoded = y_train_final
clusters = kmeans.labels_

mapping = {}
for cluster_id in range(N_CLUSTERS):
    labels_in_cluster = labels_encoded[clusters == cluster_id]
    
    if len(labels_in_cluster) > 0:
        majority_label = find_mode_numpy(labels_in_cluster)
        mapping[cluster_id] = majority_label
    else:
        mapping[cluster_id] = 0

print(mapping)

y_mapped_test_pred_encoded = np.array([mapping.get(pred, 0) for pred in y_kmeans_test_pred])

y_pred_classes_original_kmeans = encoder.inverse_transform(y_mapped_test_pred_encoded)

# generating report  
generate_final_report(y_test_original_labels=y_test, y_pred_original_labels=y_pred_classes_original_kmeans,class_names_map=ACTIVITY_NAMES)

# Ploting confusion matrix
class_labels = list(ACTIVITY_NAMES.values())
conf_matrix = confusion_matrix(y_test, y_pred_classes_original_kmeans)

fig, ax = plt.subplots(figsize=(10, 7))

im = ax.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Greens)
fig.colorbar(im)

ax.set(
    xticks=np.arange(len(class_labels)),
    yticks=np.arange(len(class_labels)),
    xticklabels=class_labels,
    yticklabels=class_labels,
    title='Confusion Matrix for K-Means Clustering (Mapped)',
    ylabel='True Activity Label',
    xlabel='Predicted Activity Label'
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = conf_matrix.max() / 2.
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        ax.text(
            j, i, conf_matrix[i, j],
            ha="center", va="center",
            color="white" if conf_matrix[i, j] > thresh else "black"
        )

plt.tight_layout()
plt.show()

#### Random Forest FineTuning

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,             
    random_state=42          
)

rf_model.fit(X_train_2d, y_train_final)
y_pred_encoded_rf = rf_model.predict(X_test_2d)
y_pred_classes_original_rf = encoder.inverse_transform(y_pred_encoded_rf)

# generating report  
generate_final_report(
    y_test_original_labels=y_test, 
    y_pred_original_labels=y_pred_classes_original_rf,
    class_names_map=ACTIVITY_NAMES
)

# Ploting confusion matrix
class_labels = list(ACTIVITY_NAMES.values())
conf_matrix = confusion_matrix(y_test, y_pred_classes_original_rf)

fig, ax = plt.subplots(figsize=(10, 7))

im = ax.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)

fig.colorbar(im)

ax.set(
    xticks=np.arange(len(class_labels)),
    yticks=np.arange(len(class_labels)),
    xticklabels=class_labels,
    yticklabels=class_labels,
    title='Confusion Matrix for Random Forest Classifier',
    ylabel='True Activity Label',
    xlabel='Predicted Activity Label'
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = conf_matrix.max() / 2.
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        ax.text(
            j, i, conf_matrix[i, j],
            ha="center", va="center",
            color="white" if conf_matrix[i, j] > thresh else "black"
        )

plt.tight_layout()
plt.show()